## Subquestion 3
To what extent can early learning behaviors, such as exercise diversity, session consis-
tency, and content focus, predict which students will struggle
or succeed later in their learning progression?

In [3]:
from utils import *
import math 
from utils import _to_unix, _ms_to_s
import matplotlib
matplotlib.use("Agg")
%load_ext autoreload
%autoreload 2

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

### 1- Load Data 

In [4]:
data_dir = r'C:\Users\fatum\Documents\EPFL\MA4\MLBD\MLBD_2026\gogymi-dataset-2025-2026-complete\out'
tables, event_tables = load_data(data_dir)

In [5]:
df_math_results = tables['math_results']
df_math_questions = tables['math_questions']
df_text_results = tables['text_results']
df_quiz_results = tables['quiz_results']
df_quiz_questions = tables['quiz_questions']
df_pageview = tables['pageviews']
df_course_ids = tables['course_ids']
df_students = tables['students']

### 2- Data Mining and Feature Engineering

First, we need to describe the engagement span of the users, to define how long our early window will be 

In [6]:
per_user, early_days = describe_platform_engagement(df_pageview, df_students, early_fraction=1/3)


Unique active days
  Mean   : 28.7
  Median : 23.0
  p25–p75: 9.0 – 42.0

Activity span (days)
  Mean   : 110.1
  Median : 122.9
  p25–p75: 56.0 – 163.8

Days since registration
  Mean   : 130.0
  Median : 134.5
  p25–p75: 89.2 – 176.5

Suggested early window (33% of median span): 41 days


All features are extracted exclusively from each student's first 41 days of engagement

In [7]:
# Filtering dfs for Early days

# Convert timestamps to dt 
df_pageview['created_at'] = pd.to_datetime(df_pageview['created_at'], errors = 'coerce')
df_math_results['timestamp'] = pd.to_datetime(df_math_results['timestamp'], errors = 'coerce')
df_text_results['timestamp'] = pd.to_datetime(df_text_results['timestamp'], errors = 'coerce')
df_quiz_results['time'] = pd.to_datetime(df_quiz_results['time'], errors = 'coerce')

In [8]:
student_meta = df_students[['user_id', 'creation_time']].drop_duplicates()
early_ts = {
    row["user_id"]: row["creation_time"] + early_days * 86400
    for _, row in df_students.iterrows()
}

In [9]:
unique_students_ids = set(df_students['user_id'])
unique_students_ids_in_pageview = set(df_pageview['user_id'])

matching_ids = unique_students_ids  & unique_students_ids_in_pageview

print(f"Matching: {len(matching_ids)}")
print(f"Original number of students: {len(unique_students_ids)}")

Matching: 1691
Original number of students: 1781


In [10]:
print("-"*20 + "Length of dfs before windowing" + "-"*20)
print(f" Pageviews {len(df_pageview)}")
print(f" Math Results {len(df_math_results)}")
print(f" Text Results {len(df_text_results)}")
print(f" Quiz Results {len(df_quiz_results)}")


# Rebuild early_days correctly from UNIX seconds, keeping UTC timezone
df_pageview['early_days'] = pd.to_datetime(df_pageview['user_id'].map(early_ts), unit='s', utc=True)
df_math_results['early_days'] = pd.to_datetime(df_math_results['user_id'].map(early_ts), unit='s')
df_text_results['early_days'] = pd.to_datetime(df_text_results['user_id'].map(early_ts), unit='s')
df_quiz_results['early_days'] = pd.to_datetime(df_quiz_results['user_id'].map(early_ts), unit='s')



# Keep only rows inside the user's first 14 days
early_pageview = df_pageview[df_pageview['created_at'] <= df_pageview['early_days']]
early_math_res = df_math_results[df_math_results['timestamp'] <= df_math_results['early_days']]
early_text_res = df_text_results[df_text_results['timestamp'] <= df_text_results['early_days']]
early_quiz_res = df_quiz_results[df_quiz_results['time'] <= df_quiz_results['early_days']]

print("-"*20 + "Length of dfs After windowing" + "-"*20)

print(f" Pageviews {len(early_pageview)}")
print(f" Math Results {len(early_math_res)}")
print(f" Text Results {len(early_text_res)}")
print(f" Quiz Results {len(early_quiz_res)}")



--------------------Length of dfs before windowing--------------------
 Pageviews 945171
 Math Results 12060
 Text Results 67351
 Quiz Results 239555
--------------------Length of dfs After windowing--------------------
 Pageviews 661760
 Math Results 10159
 Text Results 40700
 Quiz Results 224239


#### Exercise Performance
- Number of unique questions answered by each student

In [11]:
diversity_features = pd.DataFrame({'user_id': df_students['user_id'].unique()})
print(len(diversity_features))

1781


In [12]:
# Unique question ids
math_unique = early_math_res.groupby('user_id')['question_id'].nunique().rename('unique_math_qids')
text_unique = early_text_res.groupby('user_id')['question_id'].nunique().rename('unique_text_qids')
quiz_unique = early_quiz_res.groupby('user_id')['question_id'].nunique().rename('unique_quiz_qids')

diversity_features = diversity_features.merge(math_unique, on='user_id', how='left').fillna(0)
diversity_features = diversity_features.merge(text_unique, on='user_id', how='left').fillna(0)
diversity_features = diversity_features.merge(quiz_unique, on='user_id', how='left').fillna(0)
print(len(diversity_features))

diversity_features

1781


,user_id,unique_math_qids,unique_text_qids,unique_quiz_qids
0,3919,0.0,0.0,0.0
1,359,0.0,0.0,0.0
2,49,0.0,17.0,10.0
3,116,0.0,19.0,172.0
4,118,0.0,18.0,77.0
...,...,...,...,...
1776,6678,0.0,0.0,118.0
1777,6673,0.0,0.0,0.0
1778,6672,0.0,0.0,0.0
1779,6675,0.0,0.0,41.0


#### Shannon Entropy (content focus)
- Quantifies how "spread" the student's attention is accross different content categories

In [13]:
early_pageview['url']  = early_pageview['url'].astype('str') 
df_course_ids['url']  = df_course_ids['url'].astype('str') + "/"

In [14]:
tables['course_ids']

,post_id,url,post_type,course_id
0,98,/themen/einfuehrung/,lesson,42
1,100,/themen/schriftliche-addition/,lesson,42
2,102,/themen/schriftliche-subtraktion/,lesson,42
3,104,/themen/schriftliche-multiplikation/,lesson,42
4,106,/themen/schriftliche-division/,lesson,42
...,...,...,...,...
450,24762,/tests/quiz-argumentation/,quiz,3301
451,24787,/tests/quiz-erzaehlung/,quiz,3301
452,24810,/tests/quiz-erzaehlung-2/,quiz,5447
453,24823,/tests/quiz-ueberarbeitung/,quiz,3301


In [15]:
early_pageview

,id,url,user_id,created_at,early_days
25,39,/hub/,883,2025-08-18 11:13:50.464190+00:00,2025-11-04 10:25:20+00:00
28,42,/members/,883,2025-08-18 11:14:17.599904+00:00,2025-11-04 10:25:20+00:00
29,43,/hub/,883,2025-08-18 11:14:33.736479+00:00,2025-11-04 10:25:20+00:00
35,49,/kurse/,883,2025-08-18 11:16:08.128750+00:00,2025-11-04 10:25:20+00:00
42,56,/kurse/mathe-langzeitgymnasium/,883,2025-08-18 11:18:15.978771+00:00,2025-11-04 10:25:20+00:00
...,...,...,...,...,...
945131,945145,/kurse/,6872,2026-03-02 18:46:58.410131+00:00,2026-05-30 15:34:35+00:00
945166,945180,/tests/proportionen/,7007,2026-03-02 19:25:41.457196+00:00,2026-04-13 04:24:15+00:00
945167,945181,/lektionen/pruefungsmodul/,7007,2026-03-02 19:26:04.882390+00:00,2026-04-13 04:24:15+00:00
945168,945182,/themen/2021-pruefung-zap2/,7007,2026-03-02 19:26:17.943433+00:00,2026-04-13 04:24:15+00:00


In [16]:
early_pageview = early_pageview.merge(df_course_ids[['url', 'post_type']], on='url', how='left')
print(len(early_pageview))
entropy_features = early_pageview.groupby('user_id')['post_type'].apply(compute_shannon_entropy).rename('content_entropy')

661760


In [17]:
unique_course_url = set(df_course_ids['url'].astype(str))
unique_course_url_in_pageview = set(early_pageview['url'].astype(str))

matching_ids = unique_course_url  & unique_course_url_in_pageview

print(f"Matching: {len(matching_ids)}")
print(f"Original number of urls: {len(unique_course_url)}")

Matching: 403
Original number of urls: 454


#### Session Consistency (using events data)
How regularly a student engages with the platform. A student who logs in every day for short focused sessions exhibits a very different learning pattern from one who has one long burst of activity and then disappears, even if their total pageview counts are identical.
Moreover, a student who interacts a lot with the platform (lots of scrolls, pausing videos) might be less focused on the task and impact the learning outcome.
We characterize this through multiple signals: 
- **Total interaction logs** : How much the user interact with the platform
- **Idle time** : The time the user spends in between interactions with the system.
- **active days** The number of distinct calendar days with at least one recorded interaction
- **engagement span**: The time between a student's first and last activity
- **average daily pageviews** normalises volume by active days, distinguishing intensive 


In [28]:
df_features_events = extract_event_features_(event_tables, tables, early_ts)

Length before merge: 1655
Length after merge : 1659
Length before merge: 1659
Length after merge : 1659
Length before merge: 1659
Length after merge : 1659


In [29]:
outcome = build_outcome(tables, early_ts)
df_ml_events   = df_features_events.merge(outcome[["user_id", "label"]],
                                on="user_id", how="inner")
df_final = df_ml_events
dfs = [diversity_features, entropy_features]
for df in dfs: 
    df_final = df_final.merge(df, on='user_id', how='left')

df_final  = df_final.dropna(subset=["label"])


 Outcome median score: 0.587


In [30]:
# Convert timedelta to float64 (seconds)
if "heartbeat__avg_idle_time" in df_final.columns:
    if pd.api.types.is_timedelta64_dtype(df_final["heartbeat__avg_idle_time"]):
        df_final["heartbeat__avg_idle_time"] = (
            df_final["heartbeat__avg_idle_time"].dt.total_seconds().astype("float64")
        )
    else:
        df_final["heartbeat__avg_idle_time"] = pd.to_numeric(
            df_final["heartbeat__avg_idle_time"], errors="coerce"
        ).astype("float64")

df_final.dtypes

index                            int64
user_id                          int64
clicks__n_total                float64
clicks__avg_scrollY            float64
clicks__session_gap_cv         float64
clicks__scroll_depth_max       float64
heartbeat__n_total               int64
heartbeat__avg_idle_time       float64
heartbeat__scroll_depth_max    float64
heartbeat__avg_scrollY         float64
q__n_questions_viewed          float64
q__n_unique_urls               float64
q__n_unique_courses            float64
q__avg_question_number         float64
q__n_active_days               float64
q__revisit_rate                float64
media__n_play_events           float64
media__n_unique_media          float64
media__total_watch_min         float64
media__n_unique_urls           float64
media__media_type_diversity    float64
media__audio_pct               float64
media__completion_rate         float64
label                            int32
unique_math_qids               float64
unique_text_qids         

In [31]:
feature_cols = [c for c in df_final.columns if c not in ("user_id", "label")]

### Vizualisation of the feature space using KMeans and prediction with clusters as a feature

Prior work has shown that learners naturally segment into a small number of behaviorally distinct profiles.(Kizilcec, R. F., Piech, https://doi.org/10.1145/2460296.2460330). They established that unsupervised clustering makes subpopulation appear.

We apply K-Means (k=3) to our feature space and project the results onto the first two principal components to assess separability. 

In [32]:
#features_list = ['unique_math_qids', 'unique_text_qids', 'content_entropy', 'unique_quiz_qids']
X = df_final[feature_cols].fillna(0) # Handle missing values
X = X.astype('float64')
# 2. Standardize the data
scaler = StandardScaler()
groups = {
    "clicks":    ["clicks__n_total", "clicks__session_gap_cv", "clicks__scroll_depth_max"],
    "exercise":  ["unique_quiz_qids", "unique_math_qids", "unique_text_qids", "q__revisit_rate"],
    "content":   ["content_entropy", "q__n_unique_courses"],
    "media":     ["media__n_play_events", "media__completion_rate", "media__audio_pct"],
}

scaled_parts = []
for name, cols in groups.items():
    available = [c for c in cols if c in df_final.columns]
    scaled_parts.append(pd.DataFrame(
        scaler.fit_transform(df_final[available].fillna(0)),
        columns=available
    ))


X_scaled = pd.concat(scaled_parts, axis=1)


kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_final['cluster'] = clusters


pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
df_final['pca1'] = pca_results[:, 0]
df_final['pca2'] = pca_results[:, 1]


plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='pca1', y='pca2', 
    hue='cluster', 
    palette='viridis', 
    data=df_final, 
    s=100, alpha=0.7
)

plt.title('Student Behavioral Profiles: Clustering by Early Learning Metrics')
plt.xlabel('Principal Component 1 ')
plt.ylabel('Principal Component 2')
plt.legend(title='Student Group')
plt.savefig("PCA.jpg")

![Feature Importance]( \PCA.jpg "San Juan Mountains")

We observe three profiles: 
- A large moderate-engagement group (Blue) clustered around the origin
- A high-engagement group (purple) spread along the positive PC1 axis, reflecting greater exercise diversity and active days
- a small disengaged group (yellow) isolated in the negative PC1 region, likely corresponding to at-risk students. 

The clear separation of the yellow group suggests that disengaged students have a detectable behavioral signature from the earliest weeks of platform use, while the blue/purple overlap reflects ambiguity between moderate and high engagement. 

### Prediction

In [33]:
feature_cols = [c for c in df_final.columns if c not in ("user_id", "label",  "cluster", "pca1", "pca2", "index")]

In [34]:
X = df_final[feature_cols].copy()
y = df_final["label"].astype(int)
print(f"\n  Features used : {len(feature_cols)}")
print(f"  Sample size   : {len(X)}")
print(f"  Class balance : {y.value_counts().to_dict()}")


  Features used : 25
  Sample size   : 575
  Class balance : {0: 291, 1: 284}


In [35]:
results = evaluate_models(X, y, None)


── Cross-validated performance ───────────────────────────
  Model                       ROC-AUC        F1  Balanced Acc
  ------------------------------------------------------------
  Logistic Regression       0.672±0.054  0.606  0.601
  Random Forest             0.681±0.038  0.581  0.613
  Gradient Boosting         0.674±0.029  0.609  0.626
  XGBoost                   0.666±0.031  0.596  0.614


We were able to obtain better than random performance on all of our models. No single model is dominating convincingly, thus we cannot infer on the linear nature of our features' interaction. The results are then mostly influenced by the feature engineering.

In [37]:
plot_feature_importance(results, X, 'Feature_importance.png')

![Feature Importance]( Feature_importance.png "San Juan Mountains")


Through this feature importance plot, we make interesting findings:

- unique_quiz_qids and unique_math_qids rank at the top across all four models, making it a robust predictor.
- In Linear regression, unique_quiz_qids strongly predicts success : students who engage with a diverse range of quiz content early on tend to perform better
- Surprisingly, unique_math_qids strongly predicts struggle Unlike quiz diversity, math question diversity is a negative signal, suggesting students who scatter across many math topics without consolidation may be struggling to find their footing rather than exploring productively
- q_n_active_days predicts success: regular platform presence over the early window is a positive signal
- q_revisit_rate, clicks_n_total, q_n_unique_urls, q_n_questions_viewed all predict struggle, showing that high volume and repetitive revisiting are signs of difficulty, not engagement quality
- clicks_session_gap_cv predicts success, which could mean that student that interact more sparsely with the platform are more focused on solving the problems.
- Media features are consistently irrelevant
- GB and XGBoost surface content_entropy,  suggesting that it contributes through non-linear interactions 